# Problem Extractor

Estrae i problemi tecnici dai file di chat in `Chats/` usando Qwen3-32B con una finestra scorrevole di messaggi numerati.

**Flusso:**
1. Carica i messaggi da un file `.txt`
2. Passa una finestra di N messaggi numerati all'LLM
3. L'LLM identifica il primo problema completo e restituisce l'indice del prossimo messaggio da analizzare
4. Il problema estratto viene salvato come `.txt` in `Chats/problems/`
5. Il ciclo riparte dall'indice restituito

In [ ]:
from pydantic import BaseModel, Field
from llama_index.llms.openai_like import OpenAILike
from llama_index.core import PromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()

llm = OpenAILike(
    model=os.getenv("MODEL_NAME", "Qwen/Qwen3-32B-AWQ"),
    api_base=os.getenv("VLLM_API_BASE_URL", "http://10.1.2.98/v1"),
    api_key="null",
    is_chat_model=True,
    is_function_calling_model=True,
    timeout=120.0,
    temperature=0.2,
    context_window=12288,
)

class ProblemExtraction(BaseModel):
    problem_found: bool = Field(
        description=(
            "True se è stato identificato un problema/incidente tecnico completo o "
            "sufficientemente descritto nei messaggi forniti."
        )
    )
    problem_description: str = Field(
        description=(
            "Descrizione dettagliata del problema: cosa è successo, chi lo ha segnalato, "
            "quando, la causa (se identificata) e la risoluzione (se visibile nei messaggi). "
            "Stringa vuota se problem_found è False."
        )
    )
    next_start_index: int = Field(
        description=(
            "Il numero ASSOLUTO (1-based) del messaggio da cui partire per la prossima analisi. "
            "Deve essere il primo messaggio che appartiene al problema successivo "
            "o il primo non ancora analizzato."
        )
    )

print("Setup completato.")

Setup completato.


In [5]:
def load_messages(filepath: str) -> list:
    """Carica i messaggi da un file txt, filtrando le righe vuote."""
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return [line.rstrip("\n") for line in lines if line.strip()]


def format_window(messages: list, start_idx: int, window_size: int) -> str:
    """
    Formatta una finestra di messaggi con numerazione assoluta per il prompt.
    start_idx è 0-based; i numeri mostrati all'LLM sono 1-based.
    """
    end_idx = min(start_idx + window_size, len(messages))
    return "\n".join(
        f"[{start_idx + i + 1}] {msg}"
        for i, msg in enumerate(messages[start_idx:end_idx])
    )


PROMPT_TEMPLATE = PromptTemplate(
    "Sei un assistente tecnico che analizza log di chat di supporto per sistemi industriali "
    "(AGV, warehouse management, WMS, ecc.).\n\n"
    "Di seguito trovi una sequenza di messaggi numerati estratti da una chat di supporto. "
    "Il tuo compito è identificare il PRIMO problema/incidente tecnico presente in questa finestra.\n\n"
    "MESSAGGI:\n"
    "{messages}\n\n"
    "ISTRUZIONI:\n"
    "- Cerca il primo problema tecnico: errore AGV, blocco di sistema, malfunzionamento, "
    "richiesta di supporto urgente, anomalia software/hardware, ecc.\n"
    "- Un problema è sufficiente anche se non hai la risoluzione: basta la segnalazione e qualche contesto.\n"
    "- Ignora messaggi puramente amministrativi, presentazioni, saluti generici, "
    "aggiornamenti di policy senza incidente concreto.\n"
    "- `next_start_index` deve essere il numero del PRIMO messaggio che NON appartiene più "
    "al problema descritto (dove inizia il topic successivo o una nuova discussione).\n"
    "- Se non trovi nessun problema tecnico, imposta problem_found=False e "
    "next_start_index = numero dell'ultimo messaggio fornito + 1.\n"
    "- Scrivi problem_description in italiano, in modo chiaro e strutturato per un sistema RAG "
    "(includi: data/ora approssimativa, entità coinvolte, sintomo, causa, risoluzione se nota)."
)

print("Funzioni helper e prompt pronti.")

Funzioni helper e prompt pronti.


In [6]:
def extract_problems_from_file(filepath: str, output_dir: str, window_size: int = 60):
    """
    Scorre un file di chat con una finestra scorrevole, estrae ogni problema tecnico
    e lo salva come file .txt separato.

    Args:
        filepath:    percorso del file di chat sorgente
        output_dir:  cartella di destinazione per i .txt estratti
        window_size: numero di messaggi passati all'LLM per ogni chiamata
    """
    os.makedirs(output_dir, exist_ok=True)

    messages = load_messages(filepath)
    total = len(messages)
    base_name = os.path.splitext(os.path.basename(filepath))[0]
    print(f"[{base_name}] {total} messaggi caricati")

    current_idx = 0   # 0-based pointer
    problem_count = 0

    while current_idx < total:
        end_display = min(current_idx + window_size, total)
        print(f"  Analisi messaggi [{current_idx + 1} – {end_display}] ...", end=" ")

        window_text = format_window(messages, current_idx, window_size)

        try:
            result = llm.structured_predict(
                ProblemExtraction,
                PROMPT_TEMPLATE,
                messages=window_text,
            )
        except Exception as e:
            print(f"ERRORE LLM: {e}")
            current_idx += max(1, window_size // 2)
            continue

        # Converti next_start_index da 1-based a 0-based
        next_idx = result.next_start_index - 1

        if result.problem_found and result.problem_description.strip():
            problem_count += 1
            out_filename = f"{base_name}_problem_{problem_count:03d}.txt"
            out_path = os.path.join(output_dir, out_filename)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(result.problem_description)
            print(f"problema trovato → {out_filename}  (prossimo msg: {result.next_start_index})")
        else:
            print(f"nessun problema  (prossimo msg: {result.next_start_index})")

        # Sicurezza: il puntatore deve sempre avanzare
        if next_idx <= current_idx:
            print(f"  WARN: next_start_index ({result.next_start_index}) non avanza, "
                  f"forzo +{window_size // 2}")
            current_idx += max(1, window_size // 2)
        else:
            current_idx = next_idx

    print(f"  → Estratti {problem_count} problemi da {base_name}\n")
    return problem_count

In [ ]:
CHAT_FILE  = "../Chats/ChatSSI_clean.txt"
OUTPUT_DIR = "../Chats/problems"
WINDOW_SIZE = 60  # messaggi per finestra

print(f"File: {CHAT_FILE}")
total_problems = extract_problems_from_file(CHAT_FILE, OUTPUT_DIR, window_size=WINDOW_SIZE)
print(f"\nTotale problemi estratti: {total_problems}")
print(f"File salvati in: {OUTPUT_DIR}")

File: ../Chats/ChatSSI_clean.txt
[ChatSSI_clean] 21168 messaggi caricati
  Analisi messaggi [1 – 60] ... problema trovato → ChatSSI_clean_problem_001.txt  (prossimo msg: 15)
  Analisi messaggi [15 – 74] ... problema trovato → ChatSSI_clean_problem_002.txt  (prossimo msg: 19)
  Analisi messaggi [19 – 78] ... problema trovato → ChatSSI_clean_problem_003.txt  (prossimo msg: 27)
  Analisi messaggi [27 – 86] ... problema trovato → ChatSSI_clean_problem_004.txt  (prossimo msg: 40)
  Analisi messaggi [40 – 99] ... problema trovato → ChatSSI_clean_problem_005.txt  (prossimo msg: 46)
  Analisi messaggi [46 – 105] ... problema trovato → ChatSSI_clean_problem_006.txt  (prossimo msg: 73)
  Analisi messaggi [73 – 132] ... problema trovato → ChatSSI_clean_problem_007.txt  (prossimo msg: 91)
  Analisi messaggi [91 – 150] ... problema trovato → ChatSSI_clean_problem_008.txt  (prossimo msg: 144)
  Analisi messaggi [144 – 203] ... problema trovato → ChatSSI_clean_problem_009.txt  (prossimo msg: 204)
  A